In [15]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
from pyspark.sql import SparkSession
#This transforms data
from pyspark.ml.feature import VectorAssembler

In [16]:
spark = SparkSession.builder.appName("DataExploration").getOrCreate()

In [17]:
def p(data):
    pData = data.toPandas()
    print(pData.head(10))

In [18]:
df = spark.read.csv('data/diabetes_012.csv',inferSchema=True, header=True)

In [19]:
df.show(5)

+------------+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|Diabetes_012|HighBP|HighChol|CholCheck| BMI|Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex| Age|Education|Income|
+------------+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|         0.0|   1.0|     1.0|      1.0|40.0|   1.0|   0.0|                 0.0|         0.0|   0.0|    1.0|              0.0|          1.0|        0.0|    5.0|    18.0|    15.0|     1.0|0.0| 9.0|      4.0|   3.0|
|         0.0|   0.0|     0.0|      0.0|25.0|   1.0|   0.0|                 0.0|         1.0|   0.0|    0.0|              0.0|          0.0|    

In [20]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, asc

# Count occurrences of unique values in "column_name"
df.groupBy("Diabetes_012").count().sort(F.desc("count")).show()

+------------+------+
|Diabetes_012| count|
+------------+------+
|         0.0|213703|
|         2.0| 35346|
|         1.0|  4631|
+------------+------+



In [21]:
underageDrinkers = df.filter(col('Age') < 18).filter(col('HvyAlcoholConsump') == 1)
underageDrinkers.show(5)

+------------+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|Diabetes_012|HighBP|HighChol|CholCheck| BMI|Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex| Age|Education|Income|
+------------+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|         0.0|   0.0|     1.0|      1.0|28.0|   0.0|   0.0|                 0.0|         0.0|   0.0|    0.0|              1.0|          1.0|        0.0|    2.0|    10.0|     0.0|     0.0|1.0| 4.0|      6.0|   8.0|
|         0.0|   0.0|     0.0|      1.0|26.0|   0.0|   0.0|                 0.0|         1.0|   0.0|    0.0|              1.0|          1.0|    

In [28]:
underageDrinkers = underageDrinkers.select('Age', 'HvyAlcoholConsump')
underageDrinkers.show(5)

+----+-----------------+
| Age|HvyAlcoholConsump|
+----+-----------------+
| 4.0|              1.0|
| 6.0|              1.0|
|11.0|              1.0|
|10.0|              1.0|
|11.0|              1.0|
+----+-----------------+
only showing top 5 rows


In [29]:
underageDrinkers.groupBy('age').count().sort(F.desc("age")).show()

+----+-----+
| age|count|
+----+-----+
|13.0|  492|
|12.0|  692|
|11.0| 1145|
|10.0| 1782|
| 9.0| 1879|
| 8.0| 1934|
| 7.0| 1719|
| 6.0| 1241|
| 5.0|  912|
| 4.0|  821|
| 3.0|  678|
| 2.0|  520|
| 1.0|  441|
+----+-----+



In [31]:
GenderBreakdown = df.groupBy('Sex', 'Diabetes_012').count().sort(F.desc("Sex")).show()

+---+------------+------+
|Sex|Diabetes_012| count|
+---+------------+------+
|1.0|         1.0|  2027|
|1.0|         0.0| 92744|
|1.0|         2.0| 16935|
|0.0|         1.0|  2604|
|0.0|         0.0|120959|
|0.0|         2.0| 18411|
+---+------------+------+



In [40]:
pivoted_df = df.groupBy('Diabetes_012').pivot('Sex').count()
pivoted_df = pivoted_df.toDF("Diabetic Score", "Female", "Male")
pivoted_df.show()

+--------------+------+-----+
|Diabetic Score|Female| Male|
+--------------+------+-----+
|           0.0|120959|92744|
|           1.0|  2604| 2027|
|           2.0| 18411|16935|
+--------------+------+-----+

